In [1]:
from pathlib import Path
import getpass
from langchain_community import document_loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openrouter import ChatOpenRouter
from langchain_huggingface import HuggingFaceEmbeddings
import dotenv
print("-------ALL IMPORTS DONE-------")

C:\Users\vansh_hxqeh4o\AppData\Local\Temp\ipykernel_17496\3405200499.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community import document_loaders


-------ALL IMPORTS DONE-------


In [2]:
#loading the PDF file
path=r"D:\GEN AI BASICS\RAG\Vector-database\Data\llama2-research-paper.pdf"
loader = PyPDFLoader(path)
pages = loader.load()
print(f"-------LOADED {len(pages)} PAGES FROM PDF-------")

-------LOADED 77 PAGES FROM PDF-------


In [ ]:
def identify_section(paper_page):
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


In [4]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = page_document.metadata.get("page", 0)
    paper_page = page_index + 1
    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

In [5]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\GEN AI BASICS\\RAG\\Vector-database\\Data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\GEN AI BASI

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    add_start_index=True,
)
chunked_documents = text_splitter.split_documents(pages)
print(f"-------TOTAL PAGES: {len(pages)}-------")
print(f"-------SPLIT DOCUMENT INTO {len(chunked_documents)} CHUNKS-------")

-------TOTAL PAGES: 77-------
-------SPLIT DOCUMENT INTO 317 CHUNKS-------


In [7]:
for chunk_number, chunk in enumerate(chunked_documents):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [8]:
print("Chunk content:")
print(chunked_documents[0].page_content[:1000])

print("\nChunk metadata:")
print(chunked_documents[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
dimensions = embeddings_hf.embed_query("Hello, world!")
print(f"-------EMBEDDINGS DIMENSIONS: {len(dimensions)}-------")

-------EMBEDDINGS DIMENSIONS: 384-------


In [11]:
vectorstore = Chroma(
    collection_name="llama2-research-paper",
    embedding_function=embeddings_hf,
    persist_directory="D:/GEN AI BASICS/RAG/Vector-database/Data/chroma_db",
    collection_metadata={
            "hnsw:space": "cosine"
        }

)
print("-------VECTORSTORE CREATED-------")
print(f"stored chunks{len(chunked_documents)}")

-------VECTORSTORE CREATED-------
stored chunks317


In [12]:
documents=vectorstore.add_documents(chunked_documents)

In [13]:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
query="How does LLaMA 2 ensure safety in its models?"
similarity_docs = similarity_retriever.invoke(query)

In [15]:
def display_documents(documents):
    
    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:500])
        print()

In [16]:
display_documents(similarity_docs)

RANK: 1
PAPER PAGE: None
SECTION: unknown
CHUNK ID: None
SOURCE: D:\GEN AI BASICS\RAG\Vector-database\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Ethical Considerations and Limitations(Section 5.2)
Llama 2is a new technology that carries risks with use. Testing conducted to date has been in
English, and has not covered, nor could it cover all scenarios. For these reasons, as with all LLMs,
Llama 2’s potential outputs cannot be predicted in advance, and the model may in some instances
produce inaccurate or objectionable responses to user prompts. Therefore, before deploying any
applications ofLlama 2, developers should perform safety testi

RANK: 2
PAPER PAGE: None
SECTION: unknown
CHUNK ID: None
SOURCE: D:\GEN AI BASICS\RAG\Vector-database\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Ethical Considerations and Limitations(Section 5.2

In [18]:
doc.metadata.get("paper_page")

NameError: name 'doc' is not defined